![Pythonista](img/pythonista.png)

# *Terraform* para *Artifact Registry* y Google *Cloud Run*.

Esta sesión formaliza el aprovisionamiento de destino productivo en *GCP* para que el *pipeline* no dependa de infraestructura creada manualmente.

## Objetivos.

- Modelar recursos base para una ruta de despliegue en *GCP*.
- Relacionar artefactos versionados con el entorno de ejecución en prod.
- Entender el papel de Google *Artifact Registry* y *Cloud Run* como runtime gestionado.

## Puente rápido de *Terraform* (si partes de cero).

Antes de modelar recursos en *GCP*, valida este mapa mental:

- *Terraform* describe infraestructura deseada, no pasos manuales.
- `plan` te deja revisar cambios antes de ejecutar.
- `apply` solo debe correr con aprobación y contexto de entorno.
- El `state` remoto evita colisiones cuando trabajan varias personas o *pipelines*.

En esta sesión nos enfocamos en traducir componentes de despliegue (registry, service account y servicio de *Cloud Run*) a recursos declarativos versionados.

## Recursos típicos en el codebase actual.

La base de *Terraform* incluida en este repositorio modela al menos:

- un repositorio de artefactos e imágenes en Google [*Artifact Registry*](https://cloud.google.com/artifact-registry);
- una instancia de [*Cloud SQL*](https://cloud.google.com/sql) PostgreSQL;
- base y usuario de aplicación en *Cloud SQL*;
- variables y secretos separados para test y prod mediante *GitHub* Environments.

El objetivo es que la entrega continua tenga un destino reproducible y no dependiente de configuración manual.

## Relación entre artefacto, base de datos y entorno.

En esta ruta, prod vive en *GCP* con control de identidad federada por *OIDC*. El entorno *Linux* de test se trata en cuadernos posteriores como ruta didáctica de operación sobre host.

*Artifact Registry* permite mantener trazabilidad entre *commit*, *build* y despliegue sin reconstruir imagen por entorno.

**Nota:** [*Cloud Run v2*](https://cloud.google.com/run) es el runtime gestionado del curso. Consume la imagen publicada en *Artifact Registry* (o *GHCR*) y se conecta a *Cloud SQL* mediante *Auth Proxy* sin exponer puertos TCP.

## *Cloud Run v2*: ejemplo de recurso *Terraform*.

El recurso que gestiona el servicio en el stack del curso es [`google_cloud_run_v2_service`](https://registry.terraform.io/providers/hashicorp/google/latest/docs/resources/cloud_run_v2_service):

```hcl
resource "google_cloud_run_v2_service" "app" {
  name     = var.cloud_run_service_name
  location = var.region

  template {
    service_account = google_service_account.cloud_run_runtime.email

    containers {
      image = "us-docker.pkg.dev/cloudrun/container/hello"  # placeholder; el pipeline lo actualiza

      env {
        name  = "APP_ENV"
        value = var.environment
      }

      env {
        name = "DATABASE_URL"
        value_source {
          secret_key_ref {
            secret  = google_secret_manager_secret.database_url.secret_id
            version = "latest"
          }
        }
      }

      # Monta el socket de Cloud SQL Auth Proxy en /cloudsql.
      # La app se conecta por socket Unix; nunca se expone TCP directo.
      volume_mounts {
        name       = "cloudsql"
        mount_path = "/cloudsql"
      }
    }

    volumes {
      name = "cloudsql"
      cloud_sql_instance {
        instances = [google_sql_database_instance.app.connection_name]
      }
    }
  }

  lifecycle {
    # image      — la controla el pipeline de CI/CD con gcloud run deploy.
    # env        — gcloud run deploy puede modificar variables de entorno.
    # volume_mounts — gcloud añade el mount de cloudsql con --add-cloudsql-instances.
    # Sin ignore_changes en estos tres, terraform apply los revertería en cada ejecución.
    ignore_changes = [
      template[0].containers[0].image,
      template[0].containers[0].env,
      template[0].containers[0].volume_mounts,
    ]
  }
}
```

Puntos clave:

- `service_account` aplica mínimo privilegio: el contenedor corre con un SA de runtime dedicado, no con la cuenta del proyecto.
- `DATABASE_URL` se inyecta desde *Secret Manager* en tiempo de ejecución; la contraseña nunca aparece en el estado de *Terraform*.
- El bloque `volumes` + `volume_mounts` monta el socket de *Cloud SQL Auth Proxy*; la app se conecta por socket Unix en `/cloudsql/<connection_name>` sin exponer TCP.
- `lifecycle.ignore_changes` cubre `image`, `env` y `volume_mounts`: las tres partes que el pipeline de CI/CD modifica con `gcloud run deploy`. Sin esta cláusula, un `terraform apply` posterior revertirá la imagen al *placeholder* y eliminará la conexión a *Cloud SQL*.

## Puente con *runners* y despliegue.

La decisión de *runners* se mantiene consistente con el curso:

- hosted como opción por defecto para orquestar *pipeline*;
- self-hosted solo cuando una etapa requiere red privada o controles adicionales.

Esto permite sostener un mismo proceso para test *Linux* y prod *GCP* sin duplicar lógica de *build*.

## Parametrización en *Terraform*: el mismo código, distintos entornos.

*Terraform* separa la **estructura** de la infraestructura (archivos `.tf`) de los **valores** que la instancian (`terraform.tfvars` o variables de entorno). Este principio hace posible que el mismo código de infraestructura funcione en `test` y en `prod` sin duplicar recursos ni modificar el código.

### Variables de entrada.

Las variables se declaran en `variables.tf` con tipo, descripción y valor por defecto opcional:

```hcl
variable "region" {
  description = "Región de GCP donde se despliegan los recursos."
  type        = string
  default     = "us-central1"
}

variable "cloud_run_service_name" {
  description = "Nombre del servicio Cloud Run."
  type        = string
}

variable "db_password" {
  description = "Contraseña del usuario de base de datos."
  type        = string
  sensitive   = true   # Terraform no imprime este valor en logs ni en plan
}
```

Los valores se pasan en `terraform.tfvars` (nunca en el código, nunca en el repositorio si contienen secretos):

```hcl
region                 = "northamerica-south1"
cloud_run_service_name = "mi-api"
db_password            = "valor-real-aqui"
```

### Por qué esto importa en *CI/CD*.

Un *pipeline* de infraestructura que hardcodea valores en `.tf` no puede promoverse entre entornos sin editar el código — lo que rompe la trazabilidad y convierte cada entorno en un proyecto separado. Con variables:

- el mismo repositorio de *IaC* gestiona `test` y `prod`;
- los valores sensibles llegan como `TF_VAR_*` desde *GitHub* Secrets, no desde el código;
- cambiar una región o un nombre de servicio no requiere un *pull request*.

```yaml
# En el workflow de Terraform: los valores entran como variables de entorno
env:
  TF_VAR_region:                 ${{ vars.GCP_REGION }}
  TF_VAR_cloud_run_service_name: ${{ vars.GCP_CLOUD_RUN_SERVICE }}
  TF_VAR_db_password:            ${{ secrets.DB_PASSWORD }}
```

`TF_VAR_nombre` es la convención que *Terraform* usa para leer variables desde el entorno. El valor de `db_password` entra como secreto enmascarado y nunca aparece en el *plan* porque la variable está marcada con `sensitive = true`.

### Outputs: exportar lo que el pipeline necesita saber.

`outputs.tf` expone valores calculados por *Terraform* — URLs, nombres de recursos, identificadores de *service accounts* — para que el *pipeline* de la aplicación no tenga que conocer la infraestructura internamente:

```hcl
output "cloud_run_service_url" {
  description = "URL pública del servicio Cloud Run."
  value       = google_cloud_run_v2_service.app.uri
}
```

Estos valores se consultan con `terraform output -raw cloud_run_service_url` y se pueden pasar como variables a *GitHub* Environments, cerrando el ciclo: *Terraform* declara la infraestructura, los *outputs* la documentan, el *pipeline* la consume.

<p style="text-align: center"><a rel="license" href="http://creativecommons.org/licenses/by/4.0/"><img alt="Licencia Creative Commons" style="border-width:0" src="https://i.creativecommons.org/l/by/4.0/80x15.png" /></a><br />Esta obra está bajo una <a rel="license" href="http://creativecommons.org/licenses/by/4.0/">Licencia Creative Commons Atribución 4.0 Internacional</a>.</p>
<p style="text-align: center">&copy; José Luis Chiquete Valdivieso. 2017-2026.</p>